> **Status: PREPARED — NOT YET EXECUTED**  
> Paths updated 2026-06-26 after repository reorganisation.  
> Run this notebook first, before `A1_02`. No output files exist yet.

# MaCAD S.3 — Assignment 1
## A1-01: Geometry to Graph — Double-L Residential Building

**Assignment objective:** Convert a Grasshopper/Rhino residential building model into a
spatial graph using TopologicPy. Nodes represent rooms and circulation elements;
edges represent shared-face spatial adjacency.

**Course workflow:** S02 — Geometry to Topology  
**Primary reference notebooks (read-only, in `class_notebooks/S02_geometry_to_topology/`):**

| Notebook | Concept used |
|----------|--------------|
| `S02-01 Primal vs Dual.ipynb` | `CellComplex.ByCells`, `Graph.ByTopology(direct=True)` |
| `S02-03 Adjacency Vs Access.ipynb` | Aperture-based access graph (blocked — no door geometry) |
| `S02-06 Importing OBJ files.ipynb` | `Topology.ByOBJPath`, face extraction pattern |

**Student homework reviewed:** None found in repository (`01 Homework01_etm.ipynb` not present).  
Structure inferred from S02 course notebooks directly.

---

**Geometry inputs (resident_gen TB_01 exports):**

| File | Contents | Nodes produced |
|------|----------|----------------|
| `TB_01_rooms.obj` | 573 room Breps, 5 floors | room nodes |
| `TB_01_corridors.obj` | 20 corridor volumes | corridor nodes |
| `TB_01_cores.obj` | 3 core volumes | core nodes |
| `TB_01_doors.obj` | **Empty — 0 objects** | ⚠ no door apertures |
| `TB_01_metadata.csv` | Room attributes | node attributes |

> **Blocker — door geometry unavailable:**
> `TB_01_doors.obj` contains 0 objects. `Graph.ByTopology(directApertures=True)` [S02-03]
> cannot be used. Workaround: shared-face adjacency via `Graph.ByTopology(direct=True)` [S02-01].

### Assignment 1 Requirement Checklist

| # | Requirement | Status |
|---|-------------|--------|
| 1 | Grasshopper/Rhino building model referenced | TB_01 resident_gen exports |
| 2 | Room geometry loaded via `Topology.ByOBJPath` | \[ \] Run to confirm |
| 3 | Corridor and core geometry loaded | \[ \] Run to confirm |
| 4 | Architectural dictionaries assigned to all cells | \[ \] Run to confirm |
| 5 | CellComplex built from rooms + circulation | \[ \] Run to confirm |
| 6 | Spatial graph derived: `Graph.ByTopology(direct=True)` | \[ \] Run to confirm |
| 7 | Nodes = rooms + corridors + cores | \[ \] Run to confirm |
| 8 | Node attributes: space_type, node_role, apartment_id, floor_id, area, XYZ | \[ \] Run to confirm |
| 9 | Verification summary printed | \[ \] Run to confirm |
| 10 | `nodes.csv` and `edges.csv` exported to `04_graph_dataset/` | \[ \] Run to confirm |
| 11 | Graph figure saved to `05_visuals/` | \[ \] Run to confirm |

---
### Pipeline

```
TB_01_rooms.obj + corridors.obj + cores.obj
        ↓  Topology.ByOBJPath (S02-06)
Topologic Clusters (face meshes per named group)
        ↓  extract faces → SelfMerge
CellComplex  (closed volumetric cells, shared boundaries resolved)
        ↓  CellComplex.ByCells + TransferDictionariesBySelectors
Attributed CellComplex  (space_type, floor, apartment on every cell)
        ↓  Graph.ByTopology(model, direct=True)  [S02-01]
Dual Graph   (one vertex per cell, one edge per shared face)
        ↓  pandas CSV export
04_graph_dataset/nodes.csv  +  edges.csv
```

| Spatial entity | Graph role | Key attribute |
|----------------|-----------|---------------|
| Hall, Bath, Kitchen, Living, Bedroom | vertex (node_role=room) | space_type, apartment_id |
| Corridor segment | vertex (node_role=corridor) | floor_id |
| Core volume | vertex (node_role=core) | floor_id |
| Shared wall face between cells | edge | edge_type=adjacency |

---
## 1. Setup

In [1]:
# !pip install topologicpy

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print("topologicpy version:", Helper.Version())

ModuleNotFoundError: No module named 'topologicpy'

In [ ]:
# ── Renderer ───────────────────────────────────────────────────────────────
# "notebook" | "vscode" | "colab" | "browser"
renderer = "notebook"

# ── Path configuration ─────────────────────────────────────────────────────
PROJECT_ROOT = "D:/GitHub/GML_Edu"

# resident_gen exports (moved to shared/assets/ in 2026-06-26 reorganisation)
EXPORTS_DIR  = os.path.join(PROJECT_ROOT, "shared", "assets", "resident_gen", "Exports", "TB_01")

# Assignment 1 root (moved to assignments/ in 2026-06-26 reorganisation)
ASSIGN_ROOT  = os.path.join(PROJECT_ROOT, "assignments", "assignment_01_graph_generation")
OUTPUTS_DIR  = os.path.join(ASSIGN_ROOT, "04_graph_dataset")
VISUALS_DIR  = os.path.join(ASSIGN_ROOT, "05_visuals")

os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(VISUALS_DIR, exist_ok=True)

# OBJ file paths
ROOMS_OBJ  = os.path.join(EXPORTS_DIR, "TB_01_rooms.obj")
CORRS_OBJ  = os.path.join(EXPORTS_DIR, "TB_01_corridors.obj")
CORES_OBJ  = os.path.join(EXPORTS_DIR, "TB_01_cores.obj")
META_CSV   = os.path.join(EXPORTS_DIR, "TB_01_metadata.csv")

print("Path check:")
for label, path in [("Rooms OBJ   ", ROOMS_OBJ), ("Corridors OBJ", CORRS_OBJ),
                    ("Cores OBJ  ", CORES_OBJ), ("Metadata CSV ", META_CSV),
                    ("Outputs dir ", OUTPUTS_DIR), ("Visuals dir  ", VISUALS_DIR)]:
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {label}: [{status}]  {path}")

In [ ]:
# ── Demo / full-building switch ────────────────────────────────────────────
# DEMO_FLOOR = 0     → floor 0 only (~113 rooms, fast)
# DEMO_FLOOR = None  → all 5 floors (573 rooms, ~10 min)
DEMO_FLOOR = 0

if DEMO_FLOOR is not None:
    print(f"Demo mode: floor {DEMO_FLOOR} only. Set DEMO_FLOOR = None for full building.")
else:
    print("Full building mode: 5 floors, 573 rooms. This may take several minutes.")

---
## 2. Load Metadata

The metadata CSV (573 rows) carries room attributes that will be matched to topology
cells by centroid proximity after geometry import.

In [ ]:
if not os.path.exists(META_CSV):
    raise FileNotFoundError(
        f"Metadata CSV not found: {META_CSV}\n"
        "Check that EXPORTS_DIR points to shared/assets/resident_gen/Exports/TB_01/"
    )

meta_df = pd.read_csv(META_CSV)

if DEMO_FLOOR is not None:
    meta_df = meta_df[meta_df["floor_id"] == DEMO_FLOOR].reset_index(drop=True)

print(f"Metadata rows loaded : {len(meta_df)}")
print(f"Floors present       : {sorted(meta_df['floor_id'].unique())}")
print(f"Room types           : {dict(meta_df['room_type'].value_counts())}")
print(f"Zone types           : {dict(meta_df['zone_type'].value_counts())}")
print(f"Unique apartments    : {meta_df['apartment_id'].nunique()}")
meta_df.head(3)

---
## 3. Import Room Geometry

**Course pattern — S02-06 + S02-01:**
1. `Topology.ByOBJPath` → list of Cluster objects
2. Extract all faces from each Cluster
3. `Topology.SelfMerge(Cluster.ByTopologies(faces))` → resolves shared boundaries
4. `Topology.Cells(merged)` → individual closed volumetric cells

In [ ]:
if not os.path.exists(ROOMS_OBJ):
    raise FileNotFoundError(f"Rooms OBJ not found: {ROOMS_OBJ}")

# S02-06 pattern: ByOBJPath returns a list of Cluster objects
room_objs = Topology.ByOBJPath(ROOMS_OBJ, transposeAxes=False)
print(f"ByOBJPath returned {len(room_objs)} object(s) from rooms.obj")

# Extract all faces
room_faces_nested = [Topology.Faces(obj) for obj in room_objs
                     if Topology.IsInstance(obj, "Topology")]
room_faces = Helper.Flatten(room_faces_nested)
print(f"Total room faces     : {len(room_faces)}")

# SelfMerge → shared boundaries resolved → CellComplex
print("Building room CellComplex (SelfMerge — may take a moment)...")
rooms_model    = Topology.SelfMerge(Cluster.ByTopologies(room_faces))
all_room_cells = Topology.Cells(rooms_model)
print(f"Room cells extracted : {len(all_room_cells)}  (expected ~573 full / ~113 demo)")

In [ ]:
# Validate: a closed room cell must have ≥ 5 faces
valid_room_cells   = [c for c in all_room_cells if len(Topology.Faces(c)) >= 5]
invalid_room_cells = [c for c in all_room_cells if len(Topology.Faces(c)) < 5]
print(f"Valid room cells     : {len(valid_room_cells)}")
print(f"Invalid (< 5 faces)  : {len(invalid_room_cells)}  (skipped)")

# Filter to demo floor by centroid Z
if DEMO_FLOOR is not None:
    floor_z_min = DEMO_FLOOR * 3.35 + 3.0
    floor_z_max = DEMO_FLOOR * 3.35 + 7.5
    valid_room_cells = [
        c for c in valid_room_cells
        if floor_z_min <= Vertex.Z(Topology.Centroid(c)) <= floor_z_max
    ]
    print(f"Floor {DEMO_FLOOR} room cells : {len(valid_room_cells)}")

In [ ]:
# Visualise room geometry (sanity check)
Topology.Show(valid_room_cells,
              faceOpacity=0.5,
              backgroundColor="white",
              width=900, height=600,
              renderer=renderer)

---
## 4. Import Corridor and Core Geometry

In [ ]:
# ── Corridors ─────────────────────────────────────────────────────────────
corr_objs = Topology.ByOBJPath(CORRS_OBJ, transposeAxes=False)
corr_faces = Helper.Flatten([
    Topology.Faces(o) for o in corr_objs if Topology.IsInstance(o, "Topology")
])
if corr_faces:
    all_corr_cells = Topology.Cells(Topology.SelfMerge(Cluster.ByTopologies(corr_faces)))
else:
    all_corr_cells = []
    print("WARNING: no corridor faces found")
print(f"Corridor cells       : {len(all_corr_cells)}")

if DEMO_FLOOR is not None and all_corr_cells:
    all_corr_cells = [
        c for c in all_corr_cells
        if floor_z_min <= Vertex.Z(Topology.Centroid(c)) <= floor_z_max
    ]
    print(f"Floor {DEMO_FLOOR} corridor cells : {len(all_corr_cells)}")

# ── Cores ─────────────────────────────────────────────────────────────────
core_objs = Topology.ByOBJPath(CORES_OBJ, transposeAxes=False)
core_faces = Helper.Flatten([
    Topology.Faces(o) for o in core_objs if Topology.IsInstance(o, "Topology")
])
if core_faces:
    all_core_cells = Topology.Cells(Topology.SelfMerge(Cluster.ByTopologies(core_faces)))
else:
    all_core_cells = []
    print("WARNING: no core faces found")
print(f"Core cells           : {len(all_core_cells)}")

---
## 5. Assign Architectural Dictionaries

Every cell receives a `Dictionary` with the following keys:

| Key | Source | Example |
|-----|--------|---------|
| `node_name` | metadata room_id | `TB_01_F00_F0_B0_U00_Hall_01` |
| `node_role` | geometry layer | `room` / `corridor` / `core` |
| `space_type` | metadata room_type | `Hall`, `Bath`, `Corridor` |
| `zone_type` | metadata zone_type | `circulation`, `private`, `service` |
| `apartment_id` | metadata apartment_id | `TB_01_APT_000` |
| `floor_id` | metadata / Z estimate | `0` – `4` |
| `area` | metadata | m² |
| `volume` | metadata | m³ |
| `color` | lookup table | `orange`, `orchid`, etc. |

Cells are matched to metadata rows by minimum centroid distance (threshold: 2 m).

In [ ]:
SPACE_COLORS = {
    "Hall"     : "orange",
    "Bath"     : "orchid",
    "Kitchen"  : "lightcoral",
    "Living"   : "steelblue",
    "Bedroom"  : "powderblue",
    "Bedroom1" : "lightblue",
    "Bedroom2" : "lightskyblue",
    "Corridor" : "mediumseagreen",
    "Core"     : "tomato",
}

meta_coords    = meta_df[["centroid_x", "centroid_y", "centroid_z"]].values.astype(float)
MATCH_THRESHOLD = 2.0  # metres

def make_room_dict(row):
    return Dictionary.ByKeysValues(
        ["node_name","node_role","space_type","zone_type",
         "apartment_id","floor_id","area","volume","color","size"],
        [str(row["room_id"]),"room",str(row["room_type"]),str(row["zone_type"]),
         str(row["apartment_id"]),int(row["floor_id"]),
         float(row["area"]),float(row["volume"]),
         SPACE_COLORS.get(str(row["room_type"]),"white"),12]
    )

def make_circ_dict(role, space_type, floor_id, name=""):
    return Dictionary.ByKeysValues(
        ["node_name","node_role","space_type","zone_type",
         "apartment_id","floor_id","area","volume","color","size"],
        [name,role,space_type,"circulation",
         "",floor_id,0.0,0.0,
         SPACE_COLORS.get(space_type,"white"),18]
    )

def cell_floor_id(cell):
    z = Vertex.Z(Topology.Centroid(cell))
    return max(0, round((z - 4.85) / 3.35))

print("Dictionary helpers ready.")

In [ ]:
selectors       = []
matched_rooms   = 0
unmatched_cells = 0

def add_selector(cell, d):
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

# Room cells — match to metadata by centroid distance
for cell in valid_room_cells:
    c   = Topology.Centroid(cell)
    xyz = np.array([Vertex.X(c), Vertex.Y(c), Vertex.Z(c)])
    dists = np.linalg.norm(meta_coords - xyz, axis=1)
    idx   = int(np.argmin(dists))
    if dists[idx] < MATCH_THRESHOLD:
        add_selector(cell, make_room_dict(meta_df.iloc[idx]))
        matched_rooms += 1
    else:
        add_selector(cell, make_circ_dict("room", "unknown", cell_floor_id(cell)))
        unmatched_cells += 1

# Corridor cells
for i, cell in enumerate(all_corr_cells):
    add_selector(cell, make_circ_dict("corridor", "Corridor", cell_floor_id(cell),
                                     name=f"CORRIDOR_{i}"))

# Core cells
for i, cell in enumerate(all_core_cells):
    add_selector(cell, make_circ_dict("core", "Core", cell_floor_id(cell),
                                     name=f"CORE_{i}"))

print(f"Selectors created    : {len(selectors)}")
print(f"  matched rooms      : {matched_rooms}")
print(f"  corridor nodes     : {len(all_corr_cells)}")
print(f"  core nodes         : {len(all_core_cells)}")
print(f"  unmatched cells    : {unmatched_cells}")
if unmatched_cells > 5:
    print("  WARNING: many unmatched cells — check MATCH_THRESHOLD or floor Z range")

---
## 6. Build CellComplex Model

**From S02-01:** `CellComplex.ByCells([r1, r2, r3])` — the standard course method.  
Cells that share an exact face boundary are joined; others remain separate.

> **Performance:** demo (~120 cells) = seconds. Full building (596 cells) = several minutes.

In [ ]:
all_cells = [c for c in valid_room_cells + all_corr_cells + all_core_cells if c is not None]
print(f"Building CellComplex from {len(all_cells)} cells...")

model      = CellComplex.ByCells(all_cells)
model_cells = Topology.Cells(model)
print(f"CellComplex built    : {len(model_cells)} cells (type: {type(model).__name__})")
if len(model_cells) < len(all_cells) * 0.8:
    print("  WARNING: cell count lower than expected — some cells may not be valid closed solids")

In [ ]:
# Transfer dictionaries from selectors onto CellComplex cells
model = Topology.TransferDictionariesBySelectors(model, selectors, tranCells=True)

# Verify on one cell
sample = Topology.Cells(model)[0]
d_s    = Topology.Dictionary(sample)
print("Sample cell dictionary:")
print("  Keys  :", Dictionary.Keys(d_s))
print("  Values:", Dictionary.Values(d_s))

In [ ]:
# Visualise coloured massing
Topology.Show(Topology.Cells(model),
              faceColorKey="color",
              faceOpacity=0.75,
              backgroundColor="white",
              width=1000, height=700,
              renderer=renderer)

---
## 7. Build Spatial Graph

**From S02-01:** `Graph.ByTopology(house, direct=True)` — dual graph:
- One vertex per cell, placed at cell centroid
- One edge per pair of cells sharing an exact face

This is the shared-face adjacency graph — the course-standard fallback when door
apertures are unavailable. It captures within-apartment room connectivity
and corridor-to-hall connections for rooms with coincident wall geometry.

> **Note:** `Graph.ByTopology(direct=False, directApertures=True)` [S02-03] would give
> a door-access graph. This requires `TB_01_doors.obj` to contain door geometry
> — currently empty. Re-export from Grasshopper to enable this.

In [ ]:
# Build dual graph (S02-01)
graph    = Graph.ByTopology(model, direct=True)
vertices = Graph.Vertices(graph)
edges_g  = Graph.Edges(graph)

print(f"Graph nodes (vertices): {len(vertices)}")
print(f"Graph edges           : {len(edges_g)}")

In [ ]:
# Assign visual attributes to graph vertices
for v in vertices:
    d     = Topology.Dictionary(v)
    color = Dictionary.ValueAtKey(d, "color") or "white"
    size  = Dictionary.ValueAtKey(d, "size")  or 12
    d = Dictionary.SetValueAtKey(d, "color", color)
    d = Dictionary.SetValueAtKey(d, "size",  size)
    v = Topology.SetDictionary(v, d)

for e in edges_g:
    d = Dictionary.ByKeysValues(["width", "color"], [1, "#888888"])
    e = Topology.SetDictionary(e, d)

print("Visual attributes assigned to all graph elements.")

In [ ]:
# Show geometry + graph overlaid (S02-01 pattern)
Topology.Show(model, graph,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.25,
              backgroundColor="white",
              width=1000, height=700,
              renderer=renderer)

In [ ]:
# Show graph only — cleaner for reading connectivity structure
Topology.Show(graph,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              backgroundColor="white",
              width=1000, height=700,
              renderer=renderer)

---
## 8. Verification Summary

In [ ]:
import networkx as nx

G_nx = nx.Graph()
pos_to_id = {}
for i, v in enumerate(vertices):
    d  = Topology.Dictionary(v)
    st = Dictionary.ValueAtKey(d, "space_type") or "unknown"
    fl = Dictionary.ValueAtKey(d, "floor_id")
    G_nx.add_node(i, space_type=st, floor_id=fl)
    key = (round(Vertex.X(v), 2), round(Vertex.Y(v), 2), round(Vertex.Z(v), 2))
    pos_to_id[key] = i

for e in edges_g:
    sv = Edge.StartVertex(e)
    ev = Edge.EndVertex(e)
    sk = (round(Vertex.X(sv), 2), round(Vertex.Y(sv), 2), round(Vertex.Z(sv), 2))
    ek = (round(Vertex.X(ev), 2), round(Vertex.Y(ev), 2), round(Vertex.Z(ev), 2))
    ni, nj = pos_to_id.get(sk), pos_to_id.get(ek)
    if ni is not None and nj is not None and ni != nj:
        G_nx.add_edge(ni, nj)

type_counts  = {}
floor_counts = {}
for n in G_nx.nodes():
    st = G_nx.nodes[n].get("space_type", "unknown")
    fl = G_nx.nodes[n].get("floor_id") if G_nx.nodes[n].get("floor_id") is not None else -1
    type_counts[st]  = type_counts.get(st, 0) + 1
    floor_counts[fl] = floor_counts.get(fl, 0) + 1

print("=" * 54)
print("GRAPH VERIFICATION SUMMARY")
print("=" * 54)
print(f"Input cells          : {len(all_cells)}")
print(f"  rooms              : {len(valid_room_cells)}")
print(f"  corridors          : {len(all_corr_cells)}")
print(f"  cores              : {len(all_core_cells)}")
print(f"Graph nodes          : {len(vertices)}")
print(f"Graph edges          : {len(edges_g)}")
print("-" * 54)
print("Nodes by space_type:")
for k, v in sorted(type_counts.items()):  print(f"  {k:18s}: {v}")
print("-" * 54)
print("Nodes by floor:")
for k, v in sorted(floor_counts.items()): print(f"  floor {k}           : {v}")
print("-" * 54)
comps = nx.number_connected_components(G_nx)
isos  = len(list(nx.isolates(G_nx)))
print(f"Connected components : {comps}")
print(f"Isolated nodes       : {isos}")
if isos > 0:
    print("  → Isolated nodes indicate rooms with no shared-face neighbours.")
    print("    This is expected if wall geometry has non-zero thickness.")
print("=" * 54)

---
## 9. Export Graph Dataset

**Outputs:**
- `04_graph_dataset/nodes.csv` — one row per node, 12 attribute columns
- `04_graph_dataset/edges.csv` — one row per edge (src_id, dst_id, edge_type)
- `05_visuals/01_geometry_and_graph.png` — Topology.Show render

In [ ]:
# Build node DataFrame
node_rows = []
for i, v in enumerate(vertices):
    d = Topology.Dictionary(v)
    node_rows.append({
        "node_id"      : i,
        "node_name"    : Dictionary.ValueAtKey(d, "node_name")    or "",
        "node_role"    : Dictionary.ValueAtKey(d, "node_role")    or "",
        "space_type"   : Dictionary.ValueAtKey(d, "space_type")   or "",
        "zone_type"    : Dictionary.ValueAtKey(d, "zone_type")    or "",
        "apartment_id" : Dictionary.ValueAtKey(d, "apartment_id") or "",
        "floor_id"     : Dictionary.ValueAtKey(d, "floor_id"),
        "area"         : Dictionary.ValueAtKey(d, "area")         or 0.0,
        "volume"       : Dictionary.ValueAtKey(d, "volume")       or 0.0,
        "X"            : round(Vertex.X(v), 4),
        "Y"            : round(Vertex.Y(v), 4),
        "Z"            : round(Vertex.Z(v), 4),
    })
nodes_out = pd.DataFrame(node_rows)

# Build edge DataFrame
edge_rows = []
for e in edges_g:
    sv = Edge.StartVertex(e)
    ev = Edge.EndVertex(e)
    sk = (round(Vertex.X(sv), 2), round(Vertex.Y(sv), 2), round(Vertex.Z(sv), 2))
    ek = (round(Vertex.X(ev), 2), round(Vertex.Y(ev), 2), round(Vertex.Z(ev), 2))
    src = pos_to_id.get(sk)
    dst = pos_to_id.get(ek)
    if src is not None and dst is not None and src != dst:
        edge_rows.append({"src_id": src, "dst_id": dst, "edge_type": "adjacency"})
edges_out = pd.DataFrame(edge_rows)

# Write CSVs
nodes_path = os.path.join(OUTPUTS_DIR, "nodes.csv")
edges_path = os.path.join(OUTPUTS_DIR, "edges.csv")
nodes_out.to_csv(nodes_path, index=False)
edges_out.to_csv(edges_path, index=False)

print(f"nodes.csv  : {len(nodes_out)} rows  →  {nodes_path}")
print(f"edges.csv  : {len(edges_out)} rows  →  {edges_path}")
print()
print(nodes_out.head(3))

In [ ]:
# Save figure — Topology.Show returns plotly Figure when renderer=None
fig = Topology.Show(model, graph,
                    vertexSizeKey="size", vertexColorKey="color",
                    edgeWidthKey="width", edgeColorKey="color",
                    faceOpacity=0.2, backgroundColor="white",
                    width=1200, height=800, renderer=None)

if fig is not None:
    fig.show()
    try:
        fig.write_image(os.path.join(VISUALS_DIR, "01_geometry_and_graph.png"),
                        width=1200, height=800)
        print("Saved: 01_geometry_and_graph.png")
    except Exception as err:
        print(f"PNG save skipped — install kaleido for PNG export: pip install kaleido\n  ({err})")
else:
    Topology.Show(model, graph,
                  vertexSizeKey="size", vertexColorKey="color",
                  edgeWidthKey="width", edgeColorKey="color",
                  faceOpacity=0.2, backgroundColor="white",
                  width=1200, height=800, renderer=renderer)

---
## Final Submission Checklist

Before opening Notebook A1-02, confirm:

- [ ] All path checks above printed `[OK]`
- [ ] Room cell count matches expected (~113 demo / ~573 full)
- [ ] Graph nodes printed > 0
- [ ] Graph edges printed > 0
- [ ] Sample cell dictionary shows all 10 keys
- [ ] `nodes.csv` written with 12 columns
- [ ] `edges.csv` written with 3 columns
- [ ] Isolated node count understood (see verification note)
- [ ] Graph figure saved (or renderer display shown)

**Proceed to:** `A1_02_DoubleL_Graph_Visualisation_and_Export.ipynb`

**To process the full building:** set `DEMO_FLOOR = None` and re-run from cell 1.